# 3. Publication patterns, comparison, and PR3 evidence
**Questions:** How does coverage differ across outlets? Which findings depend on the pipeline?

Modern quantitative outputs run without LDA or human ratings. The primary analysis describes
this fixed corpus; it is not a held-out prediction test. Interpret small outlets, unequal date
coverage, boilerplate/transcript clusters, and keyword-filtering bias cautiously.

In [1]:
from pathlib import Path
import json, sys, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'pyproject.toml').exists())
ROOT = REPO/'final-project/code/modern-methods'
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT.parent))
from shared.corpus import CORPUS_DIR, audit_corpus, load_corpus, corpus_hash, digest, write_json
from shared import evaluation as ev
from src import topic_helpers as h, artifacts as a
OUT = ROOT/'Results/modern'
OUT.mkdir(parents=True, exist_ok=True)

In [2]:
df, embeddings = h.load_inputs(ROOT)
reservation = json.loads((OUT/'review_sample.json').read_text())
assert reservation['corpus_hash'] == corpus_hash(df)
experiments = pd.read_csv(OUT/'experiments.csv')
selection = a.selection_record(OUT, df, experiments.run_id.tolist())
texts, dictionary = ev.reference_texts(df)
notebook = json.loads((ROOT/'notebooks/02_modern_topic_modeling.ipynb').read_text())
fit_source = ''.join(''.join(c['source']) for c in notebook['cells'] if 'model-fit' in c.get('metadata',{}).get('tags',[]))
current_implementation = digest([fit_source] + [(ROOT/p).read_text() for p in ['src/artifacts.py','src/topic_helpers.py','../shared/evaluation.py']])
# Verify current data and artifacts before exporting any run's evidence.
for run_id in experiments.run_id:
    folder = OUT/'runs'/run_id
    model_check, metrics_check = a.load_run(folder)
    identity = json.loads((folder/'manifest.json').read_text())['identity']
    assert identity['corpus_hash'] == corpus_hash(df) and identity['embedding_hash'] == a.array_hash(embeddings)
    assert identity['reservation_hash'] == digest(reservation)
    if identity['implementation_hash'] != current_implementation or identity['versions'] != a.environment()['versions']:
        raise ValueError('Experiment code or environment changed; rerun notebook 02 after preserving old runs')
    assignments = pd.read_csv(folder/'assignments.csv',dtype={'article_id':str})
    assert assignments.topic_id.tolist() == model_check.topics_
    inventory = a.topic_inventory(folder, folder, len(df))
    ev.publication_exports(df, assignments, folder/'publication',dict(zip(inventory.topic_id,inventory.label)))
print('Per-run topic inventories and publication outputs are complete.')

/Users/marykate/Desktop/F26/DataMiningProject/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Per-run topic inventories and publication outputs are complete.


## Selected model and publication comparisons
CSV matrices contain all 25 publications and every topic. The main heatmap includes outlets
with at least 50 articles and the 15 largest topics plus −1. Its cells are fractions of **all**
articles in each outlet, including unassigned articles. The displayed subset need not sum to one.
These are descriptive within-corpus differences, not estimates of all coverage by an outlet.

In [3]:
modern = terms = inventory = labels = None
labels_ready = False
SELECTED = None
if selection is None:
    print('Selected-model interpretation pending: fill selection.json and rerun notebook 02.')
    display(experiments[['run_id','topic_count','coherence','outlier_fraction']])
else:
    run_id = selection['run_id']
    folder = OUT/'runs'/run_id
    SELECTED = OUT/'selected'/run_id
    SELECTED.mkdir(parents=True, exist_ok=True)
    manifest = json.loads((folder/'manifest.json').read_text())
    labels = a.labels_table(folder, SELECTED, {'run_id':run_id,'corpus_hash':corpus_hash(df),'run_identity_hash':digest(manifest['identity'])})
    labels_ready = bool(len(labels) and labels.confirmed.astype(str).str.lower().eq('true').all())
    modern = pd.read_csv(folder/'assignments.csv',dtype={'article_id':str})
    terms = pd.read_csv(folder/'terms.csv')
    inventory = a.topic_inventory(folder, SELECTED, len(df), labels)
    display(inventory[['topic_id','label','label_status','count','prevalence','coherence']])
    counts, proportions = ev.publication_exports(df, modern, SELECTED,dict(zip(inventory.topic_id,inventory.label)))
    from IPython.display import Image
    display(Image(filename=str(SELECTED/'publication_proportions.png')))
    if (SELECTED/'stability.csv').exists():
        display(pd.read_csv(SELECTED/'stability.csv'))
    else:
        print('Seed stability pending: rerun notebook 02 after selection.')
    if not labels_ready:
        print('Topic naming pending team confirmation; numeric outputs use provisional keyword labels.')

Selected-model interpretation pending: fill selection.json and rerun notebook 02.


,run_id,topic_count,coherence,outlier_fraction
0,n15_c25_s42,76,0.692091,0.202400
1,n15_c50_s42,40,0.662209,0.199264
2,n15_c100_s42,21,0.640805,0.146072
3,n30_c25_s42,70,0.704696,0.180442
4,n30_c50_s42,35,0.645542,0.190807
5,n30_c100_s42,24,0.629542,0.222040


## Laura's classical handoff
Place `assignments.csv`, `terms.csv`, `topics.csv`, and `manifest.json` in `Results/classical`.
The manifest must match the corpus hash. Assignments cover all article IDs exactly once;
terms include ten ranked unigrams per topic. Preserve LDA mixtures separately.

For independent human review, the manifest must additionally record `labels_confirmed: true`
and `review_reservation_hash` equal to the shared reservation's digest. This is Laura's
attestation that labels are frozen and the reserved articles were not development evidence.
Missing files are pending; malformed or misaligned supplied files are errors.

In [4]:
CLASSICAL = ROOT/'Results/classical'
required = ['assignments.csv','terms.csv','topics.csv','manifest.json']
missing = [name for name in required if not (CLASSICAL/name).exists()]
classical = classical_terms = classical_topics = classical_manifest = None
classical_labels_ready = False
COMPARISON = None
if missing:
    print('LDA handoff pending:', ', '.join(missing))
else:
    classical, classical_terms, classical_topics = ev.validate_classical(CLASSICAL, df.article_id.tolist())
    classical_manifest = json.loads((CLASSICAL/'manifest.json').read_text())
    if classical_manifest.get('corpus_hash') != corpus_hash(df):
        raise ValueError('LDA corpus hash differs')
    if classical_manifest.get('review_reservation_hash') not in [None, digest(reservation)]:
        raise ValueError('Classical review reservation differs from the frozen shared sample')
    classical_labels_ready = (classical_manifest.get('labels_confirmed') is True and
                              classical_manifest.get('review_reservation_hash') == digest(reservation))
    if classical_labels_ready and classical_topics.label.fillna('').str.strip().eq('').any():
        raise ValueError('Confirmed classical labels cannot be blank')

if modern is not None and classical is not None:
    COMPARISON = ROOT/'Results/comparison'/selection['run_id']
    COMPARISON.mkdir(parents=True, exist_ok=True)
    classical_coherence = ev.coherence(classical_terms, texts, dictionary)
    classical_coherence.to_csv(COMPARISON/'classical_coherence.csv',index=False)
    matrix = ev.jaccard_table(classical_terms, terms)
    matrix.to_csv(COMPARISON/'jaccard.csv')
    best = pd.DataFrame({'lda_topic':matrix.index,'modern_topic':matrix.idxmax(axis=1).values,'jaccard':matrix.max(axis=1).values})
    best['interpretation'] = np.where(best.jaccard.eq(0),'No lexical overlap; numeric tie is arbitrary','Best lexical match, not semantic equivalence')
    best.to_csv(COMPARISON/'best_lexical_matches.csv',index=False)
    fig, ax = plt.subplots(figsize=(12,8))
    image = ax.imshow(matrix,aspect='auto',vmin=0,vmax=1)
    ax.set_yticks(range(len(matrix)),matrix.index)
    ax.set_xticks(range(len(matrix.columns)),matrix.columns,rotation=90)
    ax.set(xlabel='BERTopic topic',ylabel='LDA topic',title='Top-ten unigram Jaccard overlap')
    fig.colorbar(image,ax=ax,label='Jaccard similarity')
    fig.tight_layout()
    for extension in ['png','pdf']: fig.savefig(COMPARISON/f'jaccard.{extension}',dpi=180)
    plt.show()
    ev.publication_exports(df,classical,COMPARISON/'classical',dict(zip(classical_topics.topic_id,classical_topics.label)))
    top = classical_terms.query('rank <= 10')
    modern_metrics = json.loads((OUT/'runs'/selection['run_id']/'manifest.json').read_text())['metrics']
    comparison_metrics = pd.DataFrame([
        dict(model='bertopic',**{k:modern_metrics[k] for k in ['topic_count','coherence','keyword_diversity','vocabulary_coverage','outlier_fraction']}),
        dict(model='lda',topic_count=len(classical_topics),coherence=classical_coherence.coherence.mean(),
             keyword_diversity=top.term.nunique()/len(top),vocabulary_coverage=classical_coherence.vocabulary_coverage.mean(),outlier_fraction=0)])
    comparison_metrics.to_csv(COMPARISON/'model_metrics.csv',index=False)
    display(comparison_metrics)
    paired = modern[['article_id','topic_id']].merge(classical[['article_id','topic_id']],on='article_id',validate='one_to_one',suffixes=('_modern','_lda'))
    pd.crosstab(paired.topic_id_lda,paired.topic_id_modern).to_csv(COMPARISON/'assignment_contingency.csv')
    runtime = [dict(model='bertopic',stage='fitting and representation',seconds=modern_metrics['fitting_representation_seconds']),
               dict(model='bertopic',stage='evaluation',seconds=modern_metrics['evaluation_seconds'])]
    for stage, seconds in classical_manifest.get('timings',{}).items():
        runtime.append(dict(model='lda',stage=stage,seconds=seconds))
    pd.DataFrame(runtime).to_csv(COMPARISON/'runtime_comparison.csv',index=False)
    write_json(COMPARISON/'comparison_identity.json',dict(corpus_hash=corpus_hash(df),modern_run=selection['run_id'],
               classical_run=str(classical.run_id.iloc[0]),classical_manifest_hash=digest(classical_manifest)))
else:
    print('Joint comparison pending selected modern model and validated LDA exports.')

LDA handoff pending: assignments.csv, terms.csv, topics.csv, manifest.json
Joint comparison pending selected modern model and validated LDA exports.


## Independent human evaluation
Two reviewers independently score the same reserved articles, separately for each model.
The packet includes the 1–5 rubric and qualitative flags; −1 receives no fit rating.
Labels, assignments, and the packet are frozen together, and reruns preserve reviewer edits.
No missing rating is replaced by a generated score.

Kappa intervals use 1,000 paired article bootstrap resamples. Degenerate resamples are
excluded and counted; undefined kappa remains missing with an explicit explanation.
Modern outlier counts and a separate qualitative packet accompany the assigned-article results.

In [5]:
human_status = 'Pending frozen labels and complete LDA handoff.'
if COMPARISON is not None and labels_ready and classical_labels_ready:
    lookup = {('bertopic',int(row.topic_id)):row.proposed_label for row in labels.itertuples()}
    lookup.update({('lda',int(row.topic_id)):row.label for row in classical_topics.itertuples()})
    review_folder = COMPARISON/'human_review'
    packet = ev.review_packets(df, modern, classical, review_folder, lookup,
                               article_ids=reservation['article_ids'],reservation=reservation)
    human_status = ev.summarize_reviews(review_folder)
    display(human_status)
else:
    print(human_status)
    if classical is not None and not classical_labels_ready:
        print('Laura must confirm labels and the shared review reservation in her manifest.')

Pending frozen labels and complete LDA handoff.


## Evidence for PR3 and the final report
Use the actual output tables and captions below to update the manuscript manually.
This notebook does not edit `pr3-submission.tex`. Distinguish completed quantitative results,
team interpretations, and unfinished evaluation. PR2 feedback is not present locally, so no
claim is made that it has been addressed.

- **Grid table:** Six configurations on 7,332 deduplicated articles; coherence uses a shared reference; silhouette excludes unassigned articles.
- **Topic inventory:** Topic sizes and top-ten terms; only team-confirmed labels support named thematic claims.
- **Publication heatmap:** Within-outlet shares of all articles; n>=50 outlets, 15 largest topics plus outliers; complete matrices exported separately.
- **Stability:** ARI conditional on assignment in both runs; matched publication changes plus unmatched topics and split/merge flags. Split/merge flags require two overlaps each covering at least 10% of the source/target topic.
- **Jaccard:** Lexical overlap only; LDA rows and modern columns; topic granularity can differ.
- **Runtime:** Inspect hardware and stage boundaries before comparison. Missing LDA stage timings remain unavailable.

Interpretation prompts: Which themes are supported by the example articles? Which outlets
show different shares, and could date coverage explain them? Which clusters reflect transcript
style or marginal relevance? Do matched seed topics support the same publication patterns?
Do the two pipelines tell similar stories despite different topic granularity?

In [6]:
evidence = pd.DataFrame([
    ('Data / methodology','Corpus, exclusions, embedding settings','complete','corpus_by_publication.csv; review_sample.json; cache manifest'),
    ('Experimental design / results','Six-configuration study','complete', 'experiments.csv'),
    ('Results / discussion','Team model choice','pending' if selection is None else 'complete','selection.json'),
    ('Results / discussion','Topic labels and article support','complete' if labels_ready else 'pending','selected/<run>/topic_labels.csv; topic_inventory.csv'),
    ('Results','Publication comparisons','all grid runs complete; selected interpretation pending' if selection is None else 'selected run complete','runs/<run>/publication; selected/<run>/publication_proportions.csv'),
    ('Robustness','Selected configuration seeds 7 and 21','complete' if SELECTED is not None and (SELECTED/'stability.csv').exists() else 'pending','selected/<run>/stability.csv; seed_*/publication_changes.csv'),
    ('Comparison','LDA quantitative handoff','complete' if COMPARISON is not None else 'pending Laura','Results/comparison/<run>'),
    ('Evaluation','Independent reviewer ratings','complete' if isinstance(human_status,pd.DataFrame) else 'pending','Results/comparison/<run>/human_review'),
    ('Limitations','Corpus bias, coverage, pooling, outliers','documented; team interpretation required','Notebook discussion prompts'),
    ('Response to feedback','PR2 feedback','not available in repository','Team must supply actual feedback')
],columns=['manuscript_section','evidence','status','artifact'])
evidence.to_csv(OUT/'pr3_evidence_checklist.csv',index=False)
display(evidence)
completion = pd.DataFrame([
    ('Confirm modern selection and topic labels','Needed for thematic findings','Mary Kate + team','selection rationale and confirmed labels','Team to confirm'),
    ('Run the chosen configuration with seeds 7 and 21','Assess assignment and publication-pattern stability','Mary Kate','ARI, outlier agreement, topic matches and publication changes','Team to confirm'),
    ('Provide classical handoff and frozen labels','Needed for paired comparisons','Laura','four validated exports and reservation attestation','Team to confirm'),
    ('Complete independent article ratings','Needed for interpretability evaluation','Two team reviewers','ratings and agreement summary','Team to confirm'),
    ('Held-out evaluation with rare-stratum policy','Assess transfer beyond fitted articles','Mary Kate + Laura','training-only fits and held-out metrics','Team to confirm'),
    ('Shared-period publication sensitivity','Assess unequal date-coverage effects','Mary Kate + Lauren','date-restricted comparison tables','Team to confirm'),
    ('Interpret findings, feedback, manuscript and figures','Produce a coherent conference paper','All; Lauren coordinates visualization','revised manuscript and feedback response','Team to confirm')
],columns=['remaining_task','reason','responsible_member','expected_output','completion_date'])
completion.to_csv(OUT/'completion_plan.csv',index=False)
display(completion)

,manuscript_section,evidence,status,artifact
0,Data / methodology,"Corpus, exclusions, embedding settings",complete,corpus_by_publication.csv; review_sample.json;...
1,Experimental design / results,Six-configuration study,complete,experiments.csv
2,Results / discussion,Team model choice,pending,selection.json
3,Results / discussion,Topic labels and article support,pending,selected/<run>/topic_labels.csv; topic_invento...
4,Results,Publication comparisons,all grid runs complete; selected interpretatio...,runs/<run>/publication; selected/<run>/publica...
5,Robustness,Selected configuration seeds 7 and 21,pending,selected/<run>/stability.csv; seed_*/publicati...
6,Comparison,LDA quantitative handoff,pending Laura,Results/comparison/<run>
7,Evaluation,Independent reviewer ratings,pending,Results/comparison/<run>/human_review
8,Limitations,"Corpus bias, coverage, pooling, outliers",documented; team interpretation required,Notebook discussion prompts
9,Response to feedback,PR2 feedback,not available in repository,Team must supply actual feedback


,remaining_task,reason,responsible_member,expected_output,completion_date
0,Confirm modern selection and topic labels,Needed for thematic findings,Mary Kate + team,selection rationale and confirmed labels,Team to confirm
1,Run the chosen configuration with seeds 7 and 21,Assess assignment and publication-pattern stab...,Mary Kate,"ARI, outlier agreement, topic matches and publ...",Team to confirm
2,Provide classical handoff and frozen labels,Needed for paired comparisons,Laura,four validated exports and reservation attesta...,Team to confirm
3,Complete independent article ratings,Needed for interpretability evaluation,Two team reviewers,ratings and agreement summary,Team to confirm
4,Held-out evaluation with rare-stratum policy,Assess transfer beyond fitted articles,Mary Kate + Laura,training-only fits and held-out metrics,Team to confirm
5,Shared-period publication sensitivity,Assess unequal date-coverage effects,Mary Kate + Lauren,date-restricted comparison tables,Team to confirm
6,"Interpret findings, feedback, manuscript and f...",Produce a coherent conference paper,All; Lauren coordinates visualization,revised manuscript and feedback response,Team to confirm


## Deferred analyses and responsible interpretation
The held-out experiment and shared-period sensitivity are deferred to the final-report phase.
Some publication/year strata contain one article, so strict stratification cannot put every
stratum in both partitions. Agree on and document a rare-stratum policy before constructing
the split. A genuine held-out study requires training-only vocabulary learning, fitting, and
model selection; these full-corpus exploratory results are not test performance.

No sentiment, public-opinion, or causal claims follow from these topic assignments.
Lexical overlap is not semantic identity; two team reviewers provide limited subjective evidence.
Record substantive AI assistance with coding and planning separately from human research judgments.